## Study notes for ViT
`TOPO: add related papers references by AI`

**Author: Haozhe Jia**\
This is a study note for build ViT from scratch to understand the architecture and mechanism of ViT.

In [28]:
import torch
import torch.nn as nn
import torchvision.transforms as T
from torch.optim import Adam
from torchvision.datasets.mnist import MNIST
from torch.utils.data import DataLoader
import numpy as np

In [29]:
def load_image(path, image_size=224):
    """Load an image file -> (1, 3, image_size, image_size) float tensor in [0, 1]."""
    img = Image.open(path).convert("RGB")
    transform = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
    ])
    x = transform(img).unsqueeze(0)  # add batch dim
    return x

def show_patches(x, patch_size=16):
    """BEFORE view: the original image, and the image cut into the patches
    that PatchEmbeddings' Conv2d will see."""
    img = x[0]                                      # (3, H, W)
    n = img.shape[1] // patch_size                  # patches per side, e.g. 14

    # (3, H, W) -> (3, n, n, patch_size, patch_size)
    patches = img.unfold(1, patch_size, patch_size).unfold(2, patch_size, patch_size)

    fig = plt.figure(figsize=(9, 4.5))

    ax = fig.add_subplot(1, 2, 1)
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title("Original image")
    ax.axis("off")

    # right half: n x n grid of separated tiles
    for row in range(n):
        for col in range(n):
            ax = fig.add_subplot(n, 2 * n, row * 2 * n + n + col + 1)
            ax.imshow(patches[:, row, col].permute(1, 2, 0))
            ax.axis("off")

    fig.suptitle(f"{n}x{n} = {n * n} patches of {patch_size}x{patch_size} pixels")
    plt.tight_layout()
    plt.show()

def show_embeddings(x, patch_embed, use_pca=False):
    """AFTER view: run PatchEmbeddings and visualize the output sequence."""
    with torch.no_grad():
        out = patch_embed(x)                        # (1, num_patches, hidden_size)

    print(f"input : {tuple(x.shape)}")
    print(f"output: {tuple(out.shape)}  <- sequence of {out.shape[1]} tokens, "
          f"{out.shape[2]} dims each")

    tokens = out[0]                                 # (num_patches, hidden_size)
    n = int(tokens.shape[0] ** 0.5)                 # e.g. 14

    fig, axes = plt.subplots(1, 2 if use_pca else 1, figsize=(10, 4), squeeze=False)

    ax = axes[0, 0]
    im = ax.imshow(tokens, aspect="auto", cmap="viridis")
    ax.set_title("Patch embeddings (one row = one token)")
    ax.set_xlabel("hidden dim")
    ax.set_ylabel("patch index")
    fig.colorbar(im, ax=ax)

    if use_pca:
        # project 768 dims -> 3, reshape back to the 14x14 grid, show as RGB
        _, _, v = torch.pca_lowrank(tokens, q=3)
        rgb = tokens @ v                            # (num_patches, 3)
        rgb = (rgb - rgb.min(0).values) / (rgb.max(0).values - rgb.min(0).values + 1e-8)
        ax = axes[0, 1]
        ax.imshow(rgb.reshape(n, n, 3))
        ax.set_title("Same tokens, PCA to RGB on the patch grid")
        ax.axis("off")
    plt.tight_layout()
    plt.show()
    return out

# 1. PatchEmbeddings

In [30]:
# 1. PatchEmbeddings

### In parctice

In [31]:
class PatchEmbedding(nn.Module):
  def __init__(self, d_model, img_size, patch_size, n_channels):
    super().__init__()

    self.d_model = d_model # Dimensionality of Model
    self.img_size = img_size # Image Size
    self.patch_size = patch_size # Patch Size
    self.n_channels = n_channels # Number of Channels

    self.linear_project = nn.Conv2d(self.n_channels, self.d_model, kernel_size=self.patch_size, stride=self.patch_size)

  # B: Batch Size
  # C: Image Channels
  # H: Image Height
  # W: Image Width
  # P_col: Patch Column
  # P_row: Patch Row

  def forward(self, x):
    x = self.linear_project(x) # (B, C, H, W) -> (B, d_model, P_col, P_row)

    x = x.flatten(2) # (B, d_model, P_col, P_row) -> (B, d_model, P)

    x = x.transpose(1, 2) # (B, d_model, P) -> (B, P, d_model)

    return x


## 2. Positional Embeddings
Append class token and position tokens to the patch embeddings.
The formula of Sinusoidal Positional Embeddings:

$$
PE_{(pos,\,2k)} = \sin\!\left(\frac{pos}{10000^{2k/d}}\right), \qquad
PE_{(pos,\,2k+1)} = \cos\!\left(\frac{pos}{10000^{2k/d}}\right)
$$

The `if i \% 2 == 0}` branch handles even dimensions with $\texttt{sin}$, the `else` handles odd dimensions with $\texttt{cos}$. Each position gets a fixed vector of interleaved sines and cosines at geometrically decreasing frequencies. It's $\textbf{added}$ to the token embedding at the input:

$$
x'_{pos} = \text{embedding}_{pos} + PE_{pos}
$$

In [32]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_length):
        super().__init__()
        # Create a learnable parameter named classification token
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model)) # (1, 1, d_model)

        # Creating positional encodings
        pe = torch.zeros(max_seq_length, d_model) # (max_seq_length, d_model)

        # Sinusoidal Positional Embeddings, method in paper "Attention is all you need"
        for pos in range(max_seq_length):
            for i in range(d_model):
                if i % 2 == 0:
                    pe[pos][i] = np.sin( pos / ( 10000 ** ( i / d_model )) )
                else:
                    pe[pos][i] = np.cos( pos / 10000 ** ( (i - 1) / d_model) )

        self.register_buffer('pe', pe.unsqueeze(0)) # (1, max_seq_length, d_model)

    def forward(self, x):
        # Expand to have class token for every image in batch
        # -1 in expand() means "keep this dimension as-is."
        tokens_batch = self.cls_token.expand(x.size()[0], -1, -1) # (B, 1, d_model),

        # add class tokens to the beginning of each embd
        x = torch.cat((tokens_batch, x),dim=1)

        # Adding positional encoding to embds
        x = x + self.pe # （B, max_token_length + 1, d_model)

        return x

## 3. Transformers

In [33]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        # key, query, value projection/weights for all heads, but in a batch
        self.c_attn = nn.Linear(d_model, 3 * d_model)

        # output projection
        self.c_proj = nn.Linear(d_model, d_model)        # regularization
        self.n_head = n_heads
        self.d_model = d_model

    def forward(self, x):
        B, T, C = x.size() # Batch size, sequence length, embedding dimensionality (n_embd)
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.d_model, dim=2)
        k = k.view(B,T,self.n_head,C // self.n_head).transpose(1,2) # (B, nh, T, hs)
        q = q.view(B,T,self.n_head,C // self.n_head).transpose(1,2) # (B, nh, T, hs)
        v = v.view(B,T,self.n_head,C // self.n_head).transpose(1,2) # (B, nh, T, hs)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=False)
        y = y.transpose(1,2).contiguous().view(B,T,C) # re-assemble all head outputs side by side
        # output projection
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, d_model, r_mlp):
        super().__init__()
        self.c_fc = nn.Linear(d_model, r_mlp * d_model)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(r_mlp * d_model, d_model)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class TransformerEncoder(nn.Module):
    def __init__(self, d_model, n_heads, r_mlp=4):
        super().__init__()
        self.d_model = d_model
        self.n_heads = n_heads

        # Sub-Layer 1 Normalization
        self.ln_1 = nn.LayerNorm(d_model)
        # Multi-Head Attention
        self.attn = MultiHeadAttention(d_model, n_heads)
        # Sub-Layer 2 Normalization
        self.ln_2 = nn.LayerNorm(d_model)
        # MLP layer
        self.mlp = MLP(d_model,r_mlp)

    def forward(self, x):
        # Residual Connection After Sub-Layer 1
        x = x + self.attn(self.ln_1(x))

        # Residual Connection After Sub-Layer 2
        x = x + self.mlp(self.ln_2(x))

        return x

## Vision Transformer

In [34]:
class VisionTransformer(nn.Module):
    def __init__(self, d_model, n_classes, img_size, patch_size, n_channels, n_heads, n_layers):
        super().__init__()

        assert img_size[0] % patch_size[0] == 0 and img_size[1] % patch_size[1] == 0, "img size dim must be divisible by patch_size dim"

        self.d_model = d_model # Dimensionality of model
        self.n_classes = n_classes # Number of classes
        self.img_size = img_size # Image size
        self.patch_size = patch_size # Patch size
        self.n_channels = n_channels # Number of channels
        self.n_heads = n_heads # Number of attention heads

        self.n_patches = (self.img_size[0] * self.img_size[1]) // (self.patch_size[0] * self.patch_size[1])
        self.max_seq_length = self.n_patches + 1

        self.patch_embedding = PatchEmbedding(self.d_model, self.img_size, self.patch_size, n_channels)
        self.positional_encoding = PositionalEncoding(self.d_model, self.max_seq_length)
        self.transformer_encoder = nn.Sequential(*[TransformerEncoder(self.d_model, self.n_heads) for _ in range(n_layers)])

        # Classification MLP Head
        self.classfier = nn.Sequential(
            nn.Linear(self.d_model, self.n_classes),
            nn.Softmax(dim=-1)
        )

    def forward(self, images):
        x = self.patch_embedding(images) # (B, P, dim_model)

        x = self.positional_encoding(x) # （B, max_token_length + 1, d_model)

        x = self.transformer_encoder(x)

        x = self.classfier(x[:,0])

        return x

In [35]:
d_model = 9
n_classes = 10
img_size = (32,32)
patch_size = (16,16)
n_channels = 1
n_heads = 3
n_layers = 3
batch_size = 128
epochs = 5
alpha = 0.005

transform = T.Compose([
  T.Resize(img_size),
  T.ToTensor()
])

train_set = MNIST(
  root="./../datasets", train=True, download=True, transform=transform
)
test_set = MNIST(
  root="./../datasets", train=False, download=True, transform=transform
)

train_loader = DataLoader(train_set, shuffle=True, batch_size=batch_size)
test_loader = DataLoader(test_set, shuffle=False, batch_size=batch_size)



# Training and Testing

In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", device, f"({torch.cuda.get_device_name(device)})" if torch.cuda.is_available() else "")

transformer = VisionTransformer(d_model, n_classes, img_size, patch_size, n_channels, n_heads, n_layers).to(device)

optimizer = Adam(transformer.parameters(), lr=alpha)
criterion = nn.CrossEntropyLoss()

for epoch in range(epochs):

  training_loss = 0.0
  for i, data in enumerate(train_loader, 0):
    inputs, labels = data
    inputs, labels = inputs.to(device), labels.to(device)

    optimizer.zero_grad()

    outputs = transformer(inputs)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()

    training_loss += loss.item()

  print(f'Epoch {epoch + 1}/{epochs} loss: {training_loss  / len(train_loader) :.3f}')

Using device:  cuda (NVIDIA GeForce RTX 4090)
Epoch 1/5 loss: 1.674
Epoch 2/5 loss: 1.561
Epoch 3/5 loss: 1.549
Epoch 4/5 loss: 1.541
Epoch 5/5 loss: 1.537


In [37]:
correct = 0
total = 0

with torch.no_grad():
  for data in test_loader:
    images, labels = data
    images, labels = images.to(device), labels.to(device)

    outputs = transformer(images)

    _, predicted = torch.max(outputs.data, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()
  print(f'\nModel Accuracy: {100 * correct // total} %')


Model Accuracy: 92 %
